# ?? NEXUS-9: Notebook 01 ? Dataset Audit & Anti-Leakage Verification

**Project:** NEXUS-9 Modular Research-Grade AI/ML System  
**Dataset:** FutureBench v1 (12,000 synthetic industrial supply chain scenarios)  
**Objective:** Audit schema, inspect missingness, detect outliers, explore regimes, and verify the critical anti-leakage design rule for early-warning models.


In [1]:
# Step 1: Import core dependencies and NEXUS-9 project modules
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Ensure project root is in system path for clean imports
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from nexus9.config import settings
from nexus9.data.loader import load_futurebench_data
from nexus9.data.validation import validate_schema, inspect_missingness, assert_no_data_leakage
from nexus9.data.preprocessing import Nexus9Preprocessor
from nexus9.exceptions import DataLeakageError

print('Dependencies successfully imported.')


Dependencies successfully imported.


## 1. Load Dataset and Inspect Dimensions
We load `nexus9_futurebench.csv` via the validated loader.


In [2]:
# Load the 12,000 scenario dataset
df = load_futurebench_data(PROJECT_ROOT / settings.paths.raw_data)
print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(5)


Dataset Shape: 12,000 rows x 16 columns


,scenario_id,environment_regime,economic_pressure,weather_severity,supplier_reliability,demand_shock,policy_strength,capacity,demand,supplier_delay_days,available_capacity,inventory_gap,delivery_delay_days,operating_cost,service_level,disruption_risk
0,1,normal,-1.375395,0.064330,0.877364,-0.748987,0.254262,128.969780,65.329863,1.678960,111.552061,0.000000,1.765943,1280.533616,100.000000,0
1,2,demand_surge,1.036659,0.135789,0.811151,-2.419553,0.567256,112.783377,79.581710,2.110034,88.751415,0.000000,2.561937,1377.887961,96.987377,0
2,3,normal,0.002883,0.124225,0.945947,0.601264,0.499242,108.322335,92.424692,0.000000,99.666823,0.000000,0.000000,1406.577421,100.000000,0
3,4,normal,-1.915441,0.260043,0.876038,0.182664,0.581124,72.143782,70.285459,5.650930,59.584997,10.700462,5.661705,1742.438817,90.073700,0
4,5,normal,-1.215541,0.530401,0.825962,-0.209205,0.371570,106.416076,71.005347,6.154088,77.639272,0.000000,2.351260,1467.325718,98.639487,0


## 2. Schema Validation
Ensure all expected 16 columns exist and conform to domain expectations.


In [3]:
# Execute automated schema validation
is_valid = validate_schema(df)
print(f'Schema validation passed: {is_valid}')
print('Column Data Types:')
print(df.dtypes)


Schema validation passed: True
Column Data Types:
scenario_id               int64
environment_regime          str
economic_pressure       float64
weather_severity        float64
supplier_reliability    float64
demand_shock            float64
policy_strength         float64
capacity                float64
demand                  float64
supplier_delay_days     float64
available_capacity      float64
inventory_gap           float64
delivery_delay_days     float64
operating_cost          float64
service_level           float64
disruption_risk           int64
dtype: object


## 3. Missing Value Audit
FutureBench intentionally injects missing values into selected columns (e.g., `supplier_reliability`) to reflect real-world sensor/telemetry dropouts.


In [4]:
# Audit missingness counts and percentages
missing_counts = inspect_missingness(df)
missing_pct = (missing_counts / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage (%)': missing_pct.round(2)
})
print('Missing Value Summary:')
missing_summary[missing_summary['Missing Count'] > 0]


Missing Value Summary:


,Missing Count,Missing Percentage (%)
weather_severity,180,1.5
supplier_reliability,300,2.5
operating_cost,120,1.0


## 4. Target Distribution and Regimes
Inspect the binary target `disruption_risk` (0 = Normal, 1 = Disrupted) and environmental regimes.


In [5]:
# Analyze class balance and regime frequencies
print('Target Variable Distribution (disruption_risk):')
print(df['disruption_risk'].value_counts(normalize=True).round(4) * 100)
print()
print('Environment Regime Breakdown:')
print(df['environment_regime'].value_counts())


Target Variable Distribution (disruption_risk):
disruption_risk
1    53.2
0    46.8
Name: proportion, dtype: float64

Environment Regime Breakdown:
environment_regime
normal             10067
demand_surge        1836
extreme_weather       97
Name: count, dtype: int64


## 5. Critical Anti-Leakage Audit
### The Problem:
`disruption_risk` is mathematically defined in FutureBench as:
$$\text{disruption\_risk} = \mathbb{I}(\text{delivery\_delay\_days} > 5 \;\lor\; \text{inventory\_gap} > 25 \;\lor\; \text{service\_level} < 85)$$

If an early-warning model is trained with `delivery_delay_days`, `inventory_gap`, or `service_level`, it trivially memorizes the label definition rather than learning to anticipate upcoming disruptions.

Let us demonstrate how NEXUS-9's `assert_no_data_leakage` enforces this guard:


In [6]:
# Test 1: Intentionally passing leaking columns must raise DataLeakageError
leaking_features = ['economic_pressure', 'delivery_delay_days', 'weather_severity']

try:
    assert_no_data_leakage(leaking_features)
    print('WARNING: Leakage check did NOT trigger!')
except DataLeakageError as e:
    print('SUCCESS: DataLeakageError correctly caught and prevented!')
    print(f'Error Message: {e}')

# Test 2: Clean early-warning features must pass without exception
clean_features = settings.features.numerical_features + settings.features.categorical_features
assert_no_data_leakage(clean_features)
print()
print('Clean feature set verified: No leakage detected.')


SUCCESS: DataLeakageError correctly caught and prevented!
Error Message: Data leakage detected! Features contain forbidden outcome-defining columns: ['delivery_delay_days']. Early warning models must exclude delivery_delay_days, inventory_gap, service_level, and scenario_id.

Clean feature set verified: No leakage detected.


## 6. Distribution Analysis and Outlier Detection
`operating_cost` contains a long-tail distribution with deliberate extreme outliers.


In [7]:
# Statistical summary of numerical features
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].describe().round(2).T


,count,mean,std,min,25%,50%,75%,max
scenario_id,12000.0,6000.50,3464.25,1.00,3000.75,6000.50,9000.25,12000.00
economic_pressure,12000.0,-0.02,1.00,-3.91,-0.70,-0.02,0.65,3.75
weather_severity,11820.0,0.29,0.16,0.00,0.16,0.27,0.39,0.88
supplier_reliability,11700.0,0.80,0.12,0.29,0.73,0.82,0.89,1.00
demand_shock,12000.0,-0.00,1.01,-3.94,-0.69,-0.00,0.67,4.20
policy_strength,12000.0,0.50,0.29,0.00,0.25,0.50,0.75,1.00
capacity,12000.0,100.09,17.38,70.00,84.95,100.26,115.25,130.00
demand,12000.0,95.65,19.07,26.37,82.62,95.48,108.63,162.02
supplier_delay_days,12000.0,4.64,1.92,0.00,3.32,4.57,5.90,12.91
available_capacity,12000.0,75.04,17.58,23.72,62.09,74.28,87.74,124.37


## 7. Preprocessing Demonstration
Test the `Nexus9Preprocessor` with median imputation for missing reliability, robust scaling, and one-hot encoding.


In [8]:
# Instantiate and fit preprocessor on clean early-warning feature set
from nexus9.data.splitting import get_feature_and_target_matrices

X, y = get_feature_and_target_matrices(df, include_leakage=False)
preprocessor = Nexus9Preprocessor(scaler_type='robust')
X_transformed = preprocessor.fit_transform(X)

print(f'Original Feature Shape: {X.shape}')
print(f'Transformed Preprocessed Array Shape: {X_transformed.shape}')
print(f'Transformed Feature Names ({len(preprocessor.get_feature_names_out())}):')
print(preprocessor.get_feature_names_out())
print()
print('Audit complete: Data foundation is verified and ready for Model Arena.')


Original Feature Shape: (12000, 11)
Transformed Preprocessed Array Shape: (12000, 12)
Transformed Feature Names (12):
['economic_pressure', 'weather_severity', 'supplier_reliability', 'demand_shock', 'policy_strength', 'capacity', 'demand', 'supplier_delay_days', 'available_capacity', 'operating_cost', 'environment_regime_extreme_weather', 'environment_regime_normal']

Audit complete: Data foundation is verified and ready for Model Arena.
